In [ ]:
DATA_PATH = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {DATA_PATH}\n"
    )

df = pd.read_csv(DATA_PATH)

print("Ham veri yüklendi.")
print("Veri boyutu:", df.shape)

df.head()

Ham veri yüklendi.
Veri boyutu: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df_model = df.copy()

In [4]:
if "customerID" in df_model.columns:
    df_model = df_model.drop(columns=["customerID"])

df_model.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df_model["TotalCharges"] = pd.to_numeric(df_model["TotalCharges"], errors="coerce")

missing_total_charges = df_model["TotalCharges"].isna().sum()

print("TotalCharges eksik değer sayısı:", missing_total_charges)

if missing_total_charges > 0:
    display(df.loc[df_model["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

TotalCharges eksik değer sayısı: 11


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,,No
753,0,20.25,,No
936,0,80.85,,No
1082,0,25.75,,No
1340,0,56.05,,No
3331,0,19.85,,No
3826,0,25.35,,No
4380,0,20.00,,No
5218,0,19.70,,No
6670,0,73.35,,No


In [6]:
df_model["TotalCharges"] = df_model["TotalCharges"].fillna(0)

print("TotalCharges eksik değer sayısı:", df_model["TotalCharges"].isna().sum())

TotalCharges eksik değer sayısı: 0


In [7]:
df_model["Churn"] = (df_model["Churn"] == "Yes").astype(int)

df_model["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [8]:
# 1) Ortalama aylık ödeme
# tenure sıfırsa bölme hatası olmaması için MonthlyCharges kullanılır.
df_model["AvgMonthlyCharge"] = np.where(
    df_model["tenure"] > 0,
    df_model["TotalCharges"] / df_model["tenure"],
    df_model["MonthlyCharges"]
)

# 2) İnternet hizmeti var mı?
df_model["HasInternetService"] = (df_model["InternetService"] != "No").astype(int)

# 3) Otomatik ödeme kullanıyor mu?
df_model["AutomaticPayment"] = df_model["PaymentMethod"].isin([
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]).astype(int)

# 4) Aylık sözleşme mi?
df_model["IsMonthToMonth"] = (df_model["Contract"] == "Month-to-month").astype(int)

# 5) Alınan ek internet hizmeti sayısı
internet_service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df_model["NumInternetServices"] = 0

for col in internet_service_cols:
    df_model["NumInternetServices"] += (df_model[col] == "Yes").astype(int)

# 6) Tenure grubu
df_model["TenureGroup"] = pd.cut(
    df_model["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

new_features = [
    "AvgMonthlyCharge",
    "HasInternetService",
    "AutomaticPayment",
    "IsMonthToMonth",
    "NumInternetServices",
    "TenureGroup"
]

df_model[new_features].head()

,AvgMonthlyCharge,HasInternetService,AutomaticPayment,IsMonthToMonth,NumInternetServices,TenureGroup
0,29.850000,1,0,1,1,0-12
1,55.573529,1,0,0,2,25-48
2,54.075000,1,0,1,2,0-12
3,40.905556,1,1,0,3,25-48
4,75.825000,1,0,1,0,0-12


In [9]:
feature_summary = pd.DataFrame({
    "dtype": df_model.dtypes,
    "missing": df_model.isna().sum(),
    "unique": df_model.nunique()
}).reset_index().rename(columns={"index": "column"})

feature_summary.to_csv(TABLE_DIR / "01_feature_engineering_summary.csv", index=False)

feature_summary

,column,dtype,missing,unique
0,gender,str,0,2
1,SeniorCitizen,int64,0,2
2,Partner,str,0,2
3,Dependents,str,0,2
4,tenure,int64,0,73
5,PhoneService,str,0,2
6,MultipleLines,str,0,3
7,InternetService,str,0,3
8,OnlineSecurity,str,0,3
9,OnlineBackup,str,0,3


In [10]:
df_model.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlyCharge,HasInternetService,AutomaticPayment,IsMonthToMonth,NumInternetServices,TenureGroup
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Electronic check,29.85,29.85,0,29.850000,1,0,1,1,0-12
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,Mailed check,56.95,1889.50,0,55.573529,1,0,0,2,25-48
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Mailed check,53.85,108.15,1,54.075000,1,0,1,2,0-12
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Bank transfer (automatic),42.30,1840.75,0,40.905556,1,1,0,3,25-48
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Electronic check,70.70,151.65,1,75.825000,1,0,1,0,0-12


In [11]:
PROCESSED_PATH = DATA_DIR / "telco_student_processed.csv"

df_model.to_csv(PROCESSED_PATH, index=False)

print("İşlenmiş veri seti kaydedildi:")
print(PROCESSED_PATH)
print("Veri boyutu:", df_model.shape)

İşlenmiş veri seti kaydedildi:
C:\Users\ferha\Desktop\telco-churn-analysis v2\data\telco_student_processed.csv
Veri boyutu: (7043, 26)
